In [ ]:
import kagglehub
import shutil
import os
import pandas as pd
from shapely import wkt
import geopandas as gpd

In [ ]:
def download_kaggle_file(file_name: str):

    # Clear cache
    cache_dir = os.path.expanduser("~/.cache/kagglehub")

    if os.path.exists(cache_dir):
        shutil.rmtree(cache_dir)
        print("KaggleHub cache cleared.")
    else:
        print("No cache found.")

    # 1. Download
    temp_path = kagglehub.dataset_download(file_name)
    
    # 2. Set new directory
    current_dir = os.getcwd()
    data_dir = current_dir + "/data/source_data"
    
    # 3. Move files
    print(f"Moving files from {temp_path} to {data_dir}...")
    
    files_moved = 0
    for filename in os.listdir(temp_path):
        source = os.path.join(temp_path, filename)
        destination = os.path.join(data_dir, filename)
        
        # Check if it's a file (skip subdirectories if any)
        if os.path.isfile(source):
            shutil.move(source, destination)
            print(f"Moved: {filename}")
            files_moved += 1
            
    print(f"\nFinished. {files_moved} files are now in source data folder")

In [ ]:
download_kaggle_file("kapastor/google-trends-countydma-mapping")

In [ ]:
mapping_file = "data/source_data/DMA_FIPS_County_Mapping.csv"
crosswalk_df = pd.read_csv(mapping_file)

In [ ]:
crosswalk_df.shape

In [ ]:
crosswalk_df.head()

In [ ]:
# fix portland issue
swap_map = {
        "Portland OR": "Portland-Auburn ME",
        "Portland-Auburn ME": "Portland OR"
    }
    
# Use replace to switch the values
crosswalk_df["GOOGLE_DMA"] = crosswalk_df["GOOGLE_DMA"].replace(swap_map)

In [ ]:
final_fandom_df = pd.read_csv("data/source_data/nba_team_fandom_5yr.csV")

In [ ]:
# Now merge the datasets 
# geoName (from pytrends) <-> GOOGLE_DMA (from crosswalk)
county_dma = crosswalk_df[["FIPS", "STATE", "COUNTY", "GOOGLE_DMA"]].merge(
    final_fandom_df['geoName'], 
    left_on='GOOGLE_DMA', 
    right_on='geoName', 
    how='left'
)

In [ ]:
county_dma.shape

In [ ]:
county_dma[pd.isna(county_dma["geoName"])].shape[0]

In [ ]:
# correct known problems with CT, SD, AK FIPS codes (outdated from this data)
# update SD
county_dma.loc[county_dma['FIPS'] == 46113, ['FIPS', 'COUNTY']] = [46102, 'Oglala Lakota County']

In [ ]:
# update AK
county_dma.loc[county_dma['FIPS'] == 2270, ['FIPS', 'COUNTY']] = [2158, 'Kusilvak Census Area']

alaska_split = [
    {'FIPS': 2063, 'COUNTY': 'Chugach Census Area'},
    {'FIPS': 2066, 'COUNTY': 'Copper River Census Area'}
]

old_fips = 2261
old_row = county_dma[county_dma["FIPS"] == old_fips].iloc[0]

new_rows = []
for county in alaska_split:
    new_row = old_row.copy()
    new_row['FIPS'] = county['FIPS']
    new_row['COUNTY'] = county['COUNTY']
    new_rows.append(new_row)

county_dma = county_dma[county_dma['FIPS'] != old_fips]
county_dma = pd.concat([county_dma, pd.DataFrame(new_rows)], ignore_index=True)

In [ ]:
# CT is a bit more complicated. Have to reassign 8 counties to the new 9 planning regions
ct_planning_regions = {
    "09110": {
        "name": "Capitol Planning Region",
        "primary_dma": "Hartford & New Haven CT"
    },
    "09120": {
        "name": "Greater Bridgeport Planning Region",
        "primary_dma": "New York NY",
    },
    "09130": {
        "name": "Lower Connecticut River Valley Planning Region",
        "primary_dma": "Hartford & New Haven CT"
    },
    "09140": {
        "name": "Naugatuck Valley Planning Region",
        "primary_dma": "Hartford & New Haven CT"
    },
    "09150": {
        "name": "Northeastern Connecticut Planning Region",
        "primary_dma": "Providence RI-New Bedford MA"
    },
    "09160": {
        "name": "Northwest Hills Planning Region",
        "primary_dma": "Hartford & New Haven CT"
    },
    "09170": {
        "name": "South Central Connecticut Planning Region",
        "primary_dma": "Hartford & New Haven CT"
    },
    "09180": {
        "name": "Southeastern Connecticut Planning Region",
        "primary_dma": "Hartford & New Haven CT"
    },
    "09190": {
        "name": "Western Connecticut Planning Region",
        "primary_dma": "New York NY"
    }
}

In [ ]:
# get rid of current CT rows
county_dma = county_dma[county_dma['STATE'].str.strip() != 'CT'].copy()

# Construct the new CT rows
new_ct_rows = []
for fips_code, region_data in ct_planning_regions.items():
    dma_name = region_data['primary_dma']
    
    # Build the base row
    new_row = {
        'FIPS': int(fips_code),
        'STATE': 'CT',
        'COUNTY': region_data['name'],
        'GOOGLE_DMA': dma_name,
        'geoName': dma_name
    }
            
    new_ct_rows.append(new_row)

df_new_ct = pd.DataFrame(new_ct_rows)
county_dma = pd.concat([county_dma, df_new_ct], ignore_index=True)

In [ ]:
county_dma.shape

In [ ]:
# merge with county_node data so we can align and impute fandom data
county_node = pd.read_csv("data/processed_data/county_node.csv")

In [ ]:
total_county = county_dma.merge(
    county_node, 
    left_on='FIPS', 
    right_on='geoid', 
    how='right'
)

In [ ]:
total_county.shape

In [ ]:
total_county['geometry'] = total_county['geometry'].apply(wkt.loads)
total_county = gpd.GeoDataFrame(total_county, geometry="geometry")
total_county.crs = "EPSG:4326"

In [ ]:
def plot_interactive_counties(gdf, column_to_plot='geoName'):
    # Generate Interactive Map

    tooltip_columns = ['FIPS'] + [column_to_plot]

    m = gdf.explore(
        column=column_to_plot, 
        cmap='YlGnBu',
        tiles='CartoDB Positron',
        tooltip=tooltip_columns,
        popup=True, # Clicking shows all attributes
        style_kwds={'weight': 0.5, 'color': 'white'} # Thin white borders
    )

    return m

In [ ]:
plot_interactive_counties(total_county)

In [ ]:
has_dma = total_county[total_county['geoName'].notna()].copy()
missing_dma = total_county[total_county['geoName'].isna()].copy()
has_dma = has_dma.to_crs(epsg=3857)
missing_dma = missing_dma.to_crs(epsg=3857)

# use centroids
has_dma_pts = has_dma.copy()
has_dma_pts['geometry'] = has_dma_pts.centroid

missing_dma_pts = missing_dma.drop(columns='geoName').copy()
missing_dma_pts['geometry'] = missing_dma_pts.centroid

# use nearest neighbor
joined = gpd.sjoin_nearest(
    missing_dma_pts, 
    has_dma_pts[['geoName', 'geometry']], 
    how='left', 
    rsuffix='_nearest'
)

# update gdf
total_county.loc[joined.index, 'geoName'] = joined['geoName']

In [ ]:
total_county.head()

In [ ]:
county_gdf = gpd.GeoDataFrame(total_county[["FIPS", "STATE", "COUNTY", "geoName", "geometry"]].merge(
    final_fandom_df, 
    on="geoName", 
    how="inner"
), geometry="geometry")

In [ ]:
county_gdf.head()

In [ ]:
target_cols = county_gdf.iloc[:, 5:]
county_gdf['top_category'] = target_cols.idxmax(axis=1)

In [ ]:
county_gdf.explore(
    column='top_category', 
    cmap='Set1',
    legend=True,
    tiles='CartoDB Positron',
    tooltip=['top_category', 'geoName']
)

In [ ]:
county_y = total_county[["FIPS", "STATE", "COUNTY", "geoName"]].merge(
    final_fandom_df, 
    on="geoName", 
    how="inner"
    ).rename(columns={
        'FIPS': 'geoid',
        'STATE': 'state_code',
        'COUNTY': 'county_name',
        'geoName': 'dma'
    })

county_y.columns = county_y.columns.str.lower().str.replace(' ', '_')

In [ ]:
county_y.head()

In [ ]:
county_y.to_csv('data/processed_data/county_y.csv', index=False)